[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# A Worked Design


## What you will be able to do

Take a problem and decide which parts of it become classes, what each one holds and does, and how
they relate to one another; and judge, from running code, what a design with classes buys over one
without them, and what it costs.


## The idea

### The problem

This guide has taught its tools one at a time, each in a notebook of its own: `__init__` and methods,
dunder methods, properties, class and static methods, inheritance and composition, dataclasses,
interfaces, decorators, context managers, and exceptions. Knowing each tool is not the same as knowing
which one to reach for.

Faced with a real problem, the hard question is rarely how to write a class. It is whether to write
one at all, what should go in it, and how several of them should fit together. Get that wrong and the
code still runs, but every change costs more than it should, and some mistakes that could have been
impossible stay possible.

That judgment is learned by watching it made and then making it yourself. So this notebook is three
designs and an exercise.

### What a design is

> A **design** is a decision about the shape of a program: which things become classes, what data and
> behavior each one owns, and how they relate, either one holding another or one being a kind of
> another. A good design makes the common changes cheap and the common mistakes impossible to write.

### Why it works that way

The **Why Classes** notebook gave the test that decides most of it: a class is worth writing when it
removes a way of being wrong, and when there are many of a thing, each with its own data. The nouns
in a description of the problem, such as shape, point, track and account, are the candidates for
classes. The verbs, such as calculate the area, add a track and withdraw, are the candidates for
methods.

From there, each tool in this guide answers one question. Data that must always be valid is checked
where it is created. What can be worked out is a property rather than a stored value. Behavior that
differs between kinds of a thing goes in subclasses, and parts that vary independently are held as
parts. What can go wrong gets an exception class of its own.

The first example is a whole program written both ways, once with functions and dictionaries and once
with classes, so that the difference is measured instead of claimed. The other two show classes doing
work that functions cannot do as neatly.

### Where you will meet this

In every program that outgrows a single file. The **APIs and JSON** guide ends by building a client you
would reuse, and deciding what that client holds and offers is exactly this kind of decision.

### What this notebook covers

Shapes built from points, first with functions and dictionaries, then with classes, through the same
numbered tasks. A playlist, built from dataclasses and the protocols Python calls into. A set of
notification channels, built from an abstract base class, a decorator, a context manager and a family
of exceptions. Then your turn: guidance on modeling a real-world object, and a bank account to build.

### A first look

The seed of the first example: a shape with a name, built from the points it holds. There is nothing
to run yet: read it, and read the output underneath it.

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Point:
    x: float
    y: float


class Rectangle:
    def __init__(self, name, corner, opposite):
        self.name = name
        self.points = [corner, opposite]

    def area(self):
        a, b = self.points
        return abs(b.x - a.x) * abs(b.y - a.y)


tile = Rectangle("floor tile", Point(0, 0), Point(4, 3))
print(tile.name, tile.area())
```

```
floor tile 12
```

A name, the points it is built from, and the calculation it needs. The rest of the notebook grows this
into a program.


## Setup

Four imports.

- `math` supplies `pi`, `dist`, which measures the distance between two points, and `isclose`
- `functools` supplies `wraps`, for the retry decorator in the third example
- `dataclass`, `field`, `replace` and `FrozenInstanceError` come from the `dataclasses` module, for
  points, tracks and accounts
- `ABC` and `abstractmethod` come from the `abc` module, for the shapes and the channels

**Run this cell before the rest of the notebook.**


In [1]:
import math
import functools
from dataclasses import dataclass, field, replace, FrozenInstanceError
from abc import ABC, abstractmethod

print("ready")


ready


## Worked examples

### Example 1: shapes, written with functions

The problem: a set of shapes, each with a name and built from points. Flat shapes, a rectangle, a
triangle and a circle, need their area and perimeter. Solid shapes, a box and a sphere, need their
surface area and volume. Both versions go through the same five numbered tasks:

1. Build a rectangle, a triangle, a circle, a box and a sphere.
2. Report on every shape.
3. Total the flat shapes' area, and find the solid with the largest volume.
4. Add a new kind of shape, a cylinder, and report on it.
5. Make two mistakes: misspell a kind of shape, and ask a flat shape for its volume. Each should be
   caught.

First, with functions. A point is a tuple, a shape is a dictionary with a `kind`, and each calculation
is one function that checks the kind and applies the right formula.


In [2]:
FLAT = ("rectangle", "triangle", "circle")
SOLID = ("box", "sphere")


def distance(p, q):
    return math.dist(p, q)


def area(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "rectangle":
        (x1, y1), (x2, y2) = points
        return abs(x2 - x1) * abs(y2 - y1)
    elif kind == "triangle":
        (ax, ay), (bx, by), (cx, cy) = points
        return abs((bx - ax) * (cy - ay) - (cx - ax) * (by - ay)) / 2
    elif kind == "circle":
        return math.pi * shape["radius"] ** 2


def perimeter(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "rectangle":
        (x1, y1), (x2, y2) = points
        return 2 * (abs(x2 - x1) + abs(y2 - y1))
    elif kind == "triangle":
        a, b, c = points
        return distance(a, b) + distance(b, c) + distance(c, a)
    elif kind == "circle":
        return 2 * math.pi * shape["radius"]


def surface_area(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "box":
        (x1, y1, z1), (x2, y2, z2) = points
        w, d, h = abs(x2 - x1), abs(y2 - y1), abs(z2 - z1)
        return 2 * (w * d + w * h + d * h)
    elif kind == "sphere":
        return 4 * math.pi * shape["radius"] ** 2


def volume(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "box":
        (x1, y1, z1), (x2, y2, z2) = points
        return abs(x2 - x1) * abs(y2 - y1) * abs(z2 - z1)
    elif kind == "sphere":
        return 4 / 3 * math.pi * shape["radius"] ** 3


Tasks 1 to 3.


In [3]:
# 1. Build a rectangle, a triangle, a circle, a box and a sphere.
shapes = [
    {"name": "floor tile", "kind": "rectangle", "points": [(0, 0), (4, 3)]},
    {"name": "roof gable", "kind": "triangle", "points": [(0, 0), (6, 0), (3, 4)]},
    {"name": "coaster", "kind": "circle", "points": [(0, 0)], "radius": 5},
    {"name": "crate", "kind": "box", "points": [(0, 0, 0), (2, 3, 4)]},
    {"name": "ball", "kind": "sphere", "points": [(0, 0, 0)], "radius": 1.5},
]

# 2. Report on every shape.
for shape in shapes:
    if shape["kind"] in FLAT:
        print(f"2. {shape['name']}: area {area(shape):.2f}, perimeter {perimeter(shape):.2f}")
    else:
        print(f"2. {shape['name']}: surface area {surface_area(shape):.2f}, volume {volume(shape):.2f}")

# 3. Total the flat shapes' area, and find the solid with the largest volume.
flat = [shape for shape in shapes if shape["kind"] in FLAT]
solid = [shape for shape in shapes if shape["kind"] in SOLID]
print(f"3. total flat area: {sum(area(shape) for shape in flat):.2f}")
print("3. largest volume:", max(solid, key=volume)["name"])


2. floor tile: area 12.00, perimeter 14.00
2. roof gable: area 12.00, perimeter 16.00
2. coaster: area 78.54, perimeter 31.42
2. crate: surface area 52.00, volume 24.00
2. ball: surface area 28.27, volume 14.14
3. total flat area: 102.54
3. largest volume: crate


It works, and for a program this size it is readable. Look at where each kind of shape lives, though.
The rectangle is spread across `FLAT`, `area` and `perimeter`; the sphere across `SOLID`, `surface_area`
and `volume`. Every function knows about every kind.

Task 4 shows what that costs. A cylinder is a new solid, so it has to be added in three places:
`SOLID`, `surface_area` and `volume`. A notebook cannot edit a function in place, so both functions are
written out again here with their new branch; in a file, the edit would be the same two lines added to
each.


In [4]:
SOLID = ("box", "sphere", "cylinder")


def surface_area(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "box":
        (x1, y1, z1), (x2, y2, z2) = points
        w, d, h = abs(x2 - x1), abs(y2 - y1), abs(z2 - z1)
        return 2 * (w * d + w * h + d * h)
    elif kind == "sphere":
        return 4 * math.pi * shape["radius"] ** 2
    elif kind == "cylinder":
        return 2 * math.pi * shape["radius"] * (shape["radius"] + shape["height"])


def volume(shape):
    kind, points = shape["kind"], shape["points"]
    if kind == "box":
        (x1, y1, z1), (x2, y2, z2) = points
        return abs(x2 - x1) * abs(y2 - y1) * abs(z2 - z1)
    elif kind == "sphere":
        return 4 / 3 * math.pi * shape["radius"] ** 3
    elif kind == "cylinder":
        return math.pi * shape["radius"] ** 2 * shape["height"]


# 4. Add a new kind of shape, a cylinder, and report on it.
tank = {"name": "tank", "kind": "cylinder", "points": [(0, 0, 0)], "radius": 1, "height": 3}
print(f"4. {tank['name']}: surface area {surface_area(tank):.2f}, volume {volume(tank):.2f}")


4. tank: surface area 25.13, volume 9.42


Now task 5, two mistakes of the kind that happen in every real program.


In [5]:
# 5. Make two mistakes. Each should be caught.
panel = {"name": "wall panel", "kind": "rectangel", "points": [(0, 0), (2, 5)]}
print("5. area of a misspelled kind:", area(panel))
print("5. volume of a flat shape:   ", volume(shapes[0]))


5. area of a misspelled kind: None
5. volume of a flat shape:    None


Neither mistake was caught. A misspelled kind matched no branch, and a flat shape matched no branch in
`volume`, so both functions fell off the end and returned `None`. That `None` then travels on until
something tries to add it up or compare it, and the error appears far from the typo that caused it.

An `else: raise ValueError(...)` at the end of each function would catch both, and it would have to be
remembered in every function, including every one written later.

### Example 1 again, written with classes

A point is a frozen dataclass, so it cannot be changed after it is made. `Shape` is an abstract base
class holding the name and the points. `Shape2D` and `Shape3D` declare which calculations every flat or
solid shape must provide, and each writes the report once. Every kind of shape is then one subclass,
with its own formulas.


In [6]:
@dataclass(frozen=True)
class Point:
    x: float
    y: float
    z: float = 0.0

    def distance_to(self, other):
        return math.dist((self.x, self.y, self.z), (other.x, other.y, other.z))


class Shape(ABC):
    """Something with a name, built from points."""

    def __init__(self, name, points):
        self.name = name
        self.points = list(points)

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Shape2D(Shape):
    @abstractmethod
    def area(self):
        """The area enclosed."""

    @abstractmethod
    def perimeter(self):
        """The distance around the edge."""

    def report(self):
        return f"{self.name}: area {self.area():.2f}, perimeter {self.perimeter():.2f}"


class Shape3D(Shape):
    @abstractmethod
    def surface_area(self):
        """The total area of every face."""

    @abstractmethod
    def volume(self):
        """The space enclosed."""

    def report(self):
        return f"{self.name}: surface area {self.surface_area():.2f}, volume {self.volume():.2f}"


class Rectangle(Shape2D):
    def __init__(self, name, corner, opposite):
        super().__init__(name, [corner, opposite])

    @property
    def width(self):
        return abs(self.points[1].x - self.points[0].x)

    @property
    def height(self):
        return abs(self.points[1].y - self.points[0].y)

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Triangle(Shape2D):
    def __init__(self, name, a, b, c):
        super().__init__(name, [a, b, c])

    def area(self):
        a, b, c = self.points
        return abs((b.x - a.x) * (c.y - a.y) - (c.x - a.x) * (b.y - a.y)) / 2

    def perimeter(self):
        a, b, c = self.points
        return a.distance_to(b) + b.distance_to(c) + c.distance_to(a)


class Circle(Shape2D):
    def __init__(self, name, center, radius):
        super().__init__(name, [center])
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Box(Shape3D):
    def __init__(self, name, corner, opposite):
        super().__init__(name, [corner, opposite])

    @property
    def dimensions(self):
        a, b = self.points
        return abs(b.x - a.x), abs(b.y - a.y), abs(b.z - a.z)

    def surface_area(self):
        w, d, h = self.dimensions
        return 2 * (w * d + w * h + d * h)

    def volume(self):
        w, d, h = self.dimensions
        return w * d * h


class Sphere(Shape3D):
    def __init__(self, name, center, radius):
        super().__init__(name, [center])
        self.radius = radius

    def surface_area(self):
        return 4 * math.pi * self.radius ** 2

    def volume(self):
        return 4 / 3 * math.pi * self.radius ** 3


Tasks 1 to 3.


In [7]:
# 1. Build a rectangle, a triangle, a circle, a box and a sphere.
shapes = [
    Rectangle("floor tile", Point(0, 0), Point(4, 3)),
    Triangle("roof gable", Point(0, 0), Point(6, 0), Point(3, 4)),
    Circle("coaster", Point(0, 0), 5),
    Box("crate", Point(0, 0, 0), Point(2, 3, 4)),
    Sphere("ball", Point(0, 0, 0), 1.5),
]

# 2. Report on every shape.
for shape in shapes:
    print("2.", shape.report())

# 3. Total the flat shapes' area, and find the solid with the largest volume.
flat = [shape for shape in shapes if isinstance(shape, Shape2D)]
solid = [shape for shape in shapes if isinstance(shape, Shape3D)]
print(f"3. total flat area: {sum(shape.area() for shape in flat):.2f}")
print("3. largest volume:", max(solid, key=lambda shape: shape.volume()).name)


2. floor tile: area 12.00, perimeter 14.00
2. roof gable: area 12.00, perimeter 16.00
2. coaster: area 78.54, perimeter 31.42
2. crate: surface area 52.00, volume 24.00
2. ball: surface area 28.27, volume 14.14
3. total flat area: 102.54
3. largest volume: crate


The same numbers. Task 2 became one line, because each shape reports on itself, and task 3 asks
`isinstance` instead of keeping lists of kinds up to date.

Task 4, the cylinder, is one new class. Nothing that already exists is edited.


In [8]:
class Cylinder(Shape3D):
    def __init__(self, name, center, radius, height):
        super().__init__(name, [center])
        self.radius = radius
        self.height = height

    def surface_area(self):
        return 2 * math.pi * self.radius * (self.radius + self.height)

    def volume(self):
        return math.pi * self.radius ** 2 * self.height


# 4. Add a new kind of shape, a cylinder, and report on it.
tank = Cylinder("tank", Point(0, 0, 0), 1, 3)
shapes.append(tank)
print("4.", tank.report())


4. tank: surface area 25.13, volume 9.42


The report loop, the totals and every other shape work with the cylinder unchanged, because the loop
asks each shape to report on itself and `Shape3D` guarantees the cylinder can.

Task 5, the same two mistakes.


In [9]:
# 5. Make two mistakes. Each should be caught.
try:
    Rectangel("wall panel", Point(0, 0), Point(2, 5))
except NameError as error:
    print("5. a misspelled kind of shape:", error)

try:
    shapes[0].volume()
except AttributeError as error:
    print("5. volume of a flat shape:    ", error)


5. a misspelled kind of shape: name 'Rectangel' is not defined
5. volume of a flat shape:     'Rectangle' object has no attribute 'volume'


Both mistakes were caught on the line that made them. A misspelled class name is a `NameError`,
because the kind is now a name Python checks rather than a string it compares. And a rectangle has no
`volume` method to call, so asking for one fails at once instead of quietly returning `None`.

| Task | With functions | With classes |
|---|---|---|
| 1. Build a shape | a dictionary with a `kind` string | `Rectangle("floor tile", Point(0, 0), Point(4, 3))` |
| 2. Report on a shape | an `if` on the kind, choosing which functions to call | `shape.report()` |
| 3. Pick out the solid shapes | `shape["kind"] in SOLID` | `isinstance(shape, Shape3D)` |
| 4. Add a cylinder | edit `SOLID`, `surface_area` and `volume` | write `Cylinder` |
| 5. Misspell a kind | `area` returns `None` | `NameError` on that line |
| 5. Ask a flat shape for its volume | `volume` returns `None` | `AttributeError` on that line |

The class version is longer to write. It pays that back the first time a shape is added and the first
time someone makes a mistake with one.

### When the functions are the better answer

Classes earned their place here, and it is worth being exact about why, using the **Why Classes**
notebook's signals. There are many shapes, each with its own data. The values that describe one shape
must stay together: a radius belongs to its circle, and in the dictionary version nothing stops a
radius being read from the wrong shape. And each new kind of shape has to be added somewhere, which is
one class here and three edits there.

Change the problem, and the answer changes. For one kind of shape and one calculation, a function
`rectangle_area(width, height)` is simpler than anything above, and a class would be structure with
nothing to do. Most programs contain both: classes where there are many things of several kinds, and
plain functions everywhere else.

### Example 2: a playlist

The second design is mostly data. A `Track` is a value: a title, an artist and a length, which never
changes once recorded. A `Playlist` holds tracks, refuses a duplicate, and can be measured, looped over
and searched. There is no side-by-side here; the point is to see several of this guide's tools
working together on one small, familiar thing.


In [10]:
class PlaylistError(Exception):
    """Anything that goes wrong with a playlist."""


class DuplicateTrackError(PlaylistError):
    def __init__(self, track):
        super().__init__(f"{track.title!r} by {track.artist} is already in the playlist")
        self.track = track


@dataclass(frozen=True, order=True)
class Track:
    """One recording. Two edits of the same piece count as the same track."""

    title: str
    artist: str
    seconds: int = field(compare=False)

    def __post_init__(self):
        if self.seconds <= 0:
            raise ValueError(f"a track must last at least one second, not {self.seconds}")

    @property
    def length(self):
        return f"{self.seconds // 60}:{self.seconds % 60:02d}"

    @classmethod
    def from_line(cls, line):
        title, artist, length = (part.strip() for part in line.split(" - "))
        minutes, seconds = length.split(":")
        return cls(title, artist, int(minutes) * 60 + int(seconds))


@dataclass
class Playlist:
    """An ordered list of tracks, with no track in it twice."""

    name: str
    tracks: list[Track] = field(default_factory=list, repr=False)

    def add(self, track):
        if track in self.tracks:
            raise DuplicateTrackError(track)
        self.tracks.append(track)

    def by(self, artist):
        return [track for track in self.tracks if track.artist == artist]

    @property
    def seconds(self):
        return sum(track.seconds for track in self.tracks)

    @property
    def length(self):
        minutes, seconds = divmod(self.seconds, 60)
        return f"{minutes}:{seconds:02d}"

    def __len__(self):
        return len(self.tracks)

    def __iter__(self):
        return iter(self.tracks)

    def __contains__(self, track):
        return track in self.tracks

    @classmethod
    def from_lines(cls, name, lines):
        playlist = cls(name)
        for line in lines:
            playlist.add(Track.from_line(line))
        return playlist


A playlist built from lines of text, then measured and looped over.


In [11]:
evening = Playlist.from_lines("evening", [
    "Clair de Lune - Debussy - 5:03",
    "Gymnopedie No. 1 - Satie - 3:05",
    "Arabesque No. 1 - Debussy - 4:21",
])

print(evening, "holds", len(evening), "tracks, lasting", evening.length)
for track in evening:
    print(f"  {track.title:<18} {track.artist:<8} {track.length}")


Playlist(name='evening') holds 3 tracks, lasting 12:29
  Clair de Lune      Debussy  5:03
  Gymnopedie No. 1   Satie    3:05
  Arabesque No. 1    Debussy  4:21


`Playlist.from_lines` built each `Track` with `Track.from_line`, two alternative constructors working
together. The playlist printed without its tracks, because that field has `repr=False`, and its length
is a property worked out from the tracks it holds.

Now the rules the design enforces.


In [12]:
print("sorted by title:", [track.title for track in sorted(evening)])
print("by Debussy:     ", [track.title for track in evening.by("Debussy")])

radio_edit = Track("Clair de Lune", "Debussy", 180)
print("the radio edit is already in the playlist:", radio_edit in evening)

try:
    evening.add(radio_edit)
except DuplicateTrackError as error:
    print("refused:", error, "| the edit lasts", error.track.length)

try:
    Track("Silence", "Cage", 0)
except ValueError as error:
    print("refused:", error)


sorted by title: ['Arabesque No. 1', 'Clair de Lune', 'Gymnopedie No. 1']
by Debussy:      ['Clair de Lune', 'Arabesque No. 1']
the radio edit is already in the playlist: True
refused: 'Clair de Lune' by Debussy is already in the playlist | the edit lasts 3:00
refused: a track must last at least one second, not 0


`sorted` worked because `Track` is orderable and `Playlist` can be looped over. The three-minute radio
edit counts as the same track as the full recording, because `seconds` is left out of the comparison,
so `in` found it and `add` refused it with an exception carrying the rejected track. A track with no
length never got as far as a playlist.

| In the playlist | The capability | Where this guide taught it |
|---|---|---|
| `Track` written from three fields | `@dataclass` | **Dataclasses** |
| a radio edit counted as the same track | `field(compare=False)` | **Dataclasses** |
| tracks that sort by title | `order=True` | **Dataclasses** |
| a track of no length refused | `__post_init__` | **Dataclasses** |
| `track.length` and `playlist.length` | computed properties | **Properties** |
| `Track.from_line` and `Playlist.from_lines` | alternative constructors | **Class and Static Methods** |
| `len`, `for` and `in` on a playlist | `__len__`, `__iter__` and `__contains__` | **Dunder Methods** and **Context Managers and Iterators** |
| a playlist holding its tracks | composition | **Composition over Inheritance** |
| `DuplicateTrackError` carrying the track | an exception with attributes | **Exceptions as Classes** |

### Example 3: notification channels

The third design is mostly behavior. A message can go out by email, by text or by pager, and the
program sending it should not care which. Each channel is a kind of `Channel`, which fixes what every
channel must do. The pager works over a network that drops requests, so its delivery retries. And an
`Outbox` collects messages and sends them all at once, or none of them if something goes wrong first.


In [13]:
class DeliveryError(Exception):
    """Anything that stops a message being delivered."""


class MessageTooLongError(DeliveryError):
    def __init__(self, channel, length, limit):
        super().__init__(f"{channel.name}: {length} characters, and the limit is {limit}")
        self.length = length
        self.limit = limit


def retry(times):
    """Try again after a ConnectionError, up to times attempts in all."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    print(f"    attempt {attempt} of {times} failed: {error}")
            raise DeliveryError(f"gave up after {times} attempts")
        return wrapper
    return decorator


class Channel(ABC):
    """A way of sending a message. Each subclass says how."""

    def __init__(self, name):
        self.name = name
        self.sent = []

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

    def send(self, recipient, message):
        self.check(message)
        self.deliver(recipient, message)
        self.sent.append((recipient, message))

    def check(self, message):
        """Refuse a message this channel cannot carry. By default, nothing is refused."""

    @abstractmethod
    def deliver(self, recipient, message):
        """Actually send the message."""


class Email(Channel):
    def deliver(self, recipient, message):
        print(f"    email to {recipient}: {message}")


class SMS(Channel):
    LIMIT = 160

    def check(self, message):
        if len(message) > self.LIMIT:
            raise MessageTooLongError(self, len(message), self.LIMIT)

    def deliver(self, recipient, message):
        print(f"    text to {recipient}: {message}")


class Pager(Channel):
    """Reaches the engineer on call, over a network that drops the first few requests."""

    def __init__(self, name, drops):
        super().__init__(name)
        self.drops = drops

    @retry(times=3)
    def deliver(self, recipient, message):
        if self.drops > 0:
            self.drops -= 1
            raise ConnectionError("no answer from the pager network")
        print(f"    page to {recipient}: {message}")


class Outbox:
    """Queues messages inside a with block, and sends all of them at the end or none."""

    def __init__(self):
        self.queue = []

    def add(self, channel, recipient, message):
        self.queue.append((channel, recipient, message))

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            for channel, recipient, message in self.queue:
                channel.send(recipient, message)
        else:
            print(f"    nothing sent: {len(self.queue)} queued, and the block raised {exc_type.__name__}")
        return False


One message through every channel, then the rules.


In [14]:
channels = [Email("office email"), SMS("phone"), Pager("on-call pager", drops=1)]

for channel in channels:
    print(f"{channel!r}:")
    channel.send("duty officer", "Tromso reading overdue")

try:
    channels[1].send("duty officer", "x" * 200)
except MessageTooLongError as error:
    print("refused:", error, "| over by", error.length - error.limit)

try:
    Channel("generic")
except TypeError as error:
    print("refused:", error)


Email('office email'):
    email to duty officer: Tromso reading overdue
SMS('phone'):
    text to duty officer: Tromso reading overdue
Pager('on-call pager'):
    attempt 1 of 3 failed: no answer from the pager network
    page to duty officer: Tromso reading overdue
refused: phone: 200 characters, and the limit is 160 | over by 40
refused: Can't instantiate abstract class Channel without an implementation for abstract method 'deliver'


One loop sent through three different channels, and each did it its own way. The pager's first attempt
failed and `@retry` tried again, without `Pager.deliver` containing a line about retrying. The text was
refused before sending by `SMS.check`, which overrides a method that `Channel.send` calls for every
channel. And `Channel` itself cannot be created, because it leaves `deliver` to its subclasses.

Now the outbox, once when everything goes well and once when the block fails.


In [15]:
print("a block that finishes:")
with Outbox() as outbox:
    outbox.add(channels[0], "team", "daily summary ready")
    outbox.add(channels[1], "duty officer", "all stations reporting")

print("a block that fails:")
try:
    with Outbox() as outbox:
        outbox.add(channels[0], "team", "daily summary ready")
        raise RuntimeError("the summary failed to build")
except RuntimeError:
    pass

print("office email has now sent", len(channels[0].sent), "messages")

dead = Pager("dead pager", drops=5)
try:
    dead.send("duty officer", "anyone there?")
except DeliveryError as error:
    print("refused:", error, "| recorded as sent:", len(dead.sent))


a block that finishes:
    email to team: daily summary ready
    text to duty officer: all stations reporting
a block that fails:
    nothing sent: 1 queued, and the block raised RuntimeError
office email has now sent 2 messages
    attempt 1 of 3 failed: no answer from the pager network
    attempt 2 of 3 failed: no answer from the pager network
    attempt 3 of 3 failed: no answer from the pager network
refused: gave up after 3 attempts | recorded as sent: 0


The first block sent both queued messages as it ended. The second queued one and then raised, so the
outbox sent nothing and let the error continue to the `except`. Office email's count confirms it: one
message from the first loop, one from the first outbox, and none from the second. A pager that never
answered gave up after three attempts with a `DeliveryError`, and nothing was recorded as sent.

| In the channels | The capability | Where this guide taught it |
|---|---|---|
| `Channel()` refused, and every channel required to write `deliver` | an abstract base class | **Interfaces** |
| `Email`, `SMS` and `Pager` built on `Channel` | inheritance and overriding | **Inheritance** |
| `Channel.send` calling `check` and `deliver` for any channel | shared code calling methods a subclass supplies | **Interfaces** |
| `LIMIT = 160` | a class attribute | **Class and Static Methods** |
| `Pager.__init__` calling `super().__init__` | extending `__init__` | **Inheritance** |
| `@retry(times=3)` on `deliver` | a decorator with settings, on a method | **Decorators** |
| `Outbox` sending all or nothing | a context manager | **Context Managers and Iterators** |
| `MessageTooLongError` with `length` and `limit` | an exception family with attributes | **Exceptions as Classes** |
| `Pager('on-call pager')` in the output | `__repr__` using `type(self).__name__` | **Dunder Methods** |
| one loop sending through every channel | polymorphism | **Inheritance** |


## Your turn

The three examples started from a problem and asked the same questions each time. Ask them of any
real-world object you want to model:

1. *What are the things?* The nouns in a description of the problem. Each is a candidate class.
2. *What does each one hold?* Its fields. Which of them must always be valid, and where is that
   checked?
3. *What does each one do?* The verbs. Which change the object, and which only work something out?
4. *What can be worked out rather than stored?* Those are properties.
5. *How do they relate?* One holding another is composition. One being a kind of another is
   inheritance.
6. *What can go wrong?* Each failure a caller would handle differently gets an exception class.
7. *Where does the data come from?* A file or a web service suggests an alternative constructor.

The six tasks below model a bank account by asking those questions in order. Write your answer in the
cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/14-a-worked-design-solutions.ipynb).

**1.** Write `Transaction`, a frozen dataclass with `amount` and `description`, whose `__post_init__`
refuses an amount of zero. Create a deposit of `100.0` and a withdrawal of `-35.5`.


In [16]:
# your code here


**2.** Write `Account`, a dataclass that holds an `owner` and a list of transactions, with `deposit`
and `withdraw` methods that add transactions, and a read-only `balance` property worked out from them.


In [17]:
# your code here


**3.** Write `AccountError` and a subclass, `InsufficientFundsError`, that carries `balance` and
`requested`. Make `withdraw` raise it when the money is not there, then catch it and print how much is
missing.


In [18]:
# your code here


**4.** Make `len(account)` count the transactions and `for` loop over them, and give `Account` a
`__repr__` that shows the owner and the balance, using `type(self).__name__`.


In [19]:
# your code here


**5.** Write a class method `from_statement(owner, lines)` that builds an account from lines such as
`"+250.00 salary"` and `"-12.40 lunch"`.


In [20]:
# your code here


**6.** Write `SavingsAccount(Account)`, which allows at most two withdrawals, raising a new
`TooManyWithdrawalsError` on the third, and has an `add_interest(rate)` method. Use `super()` to reuse
`Account.withdraw`.


In [21]:
# your code here


When the account works, model something else the same way, with no solutions to lean on. Any of these
repays the effort:

- **A library**: books, members and loans. A loan belongs to one member and one book, and a book
  cannot be lent twice at once.
- **A recipe**: ingredients with quantities, and a recipe that scales itself to a different number
  of servings.
- **A parking garage**: vehicles of different sizes, spaces that fit some of them, and tickets that
  record when a vehicle arrived.
- **A to-do list**: tasks with priorities and due dates, which sort, filter and can be marked done.
- **A shop**: products, stock levels, and orders that cannot take more than is in stock.

Start with the seven questions, write the answers down before any code, and expect to change the design
at least once after the first version runs.


## Common errors

### TypeError: a new kind of shape that forgets a calculation

`Shape3D` requires both `surface_area` and `volume`. A new solid that writes only one is refused when it
is created.


In [22]:
class Cone(Shape3D):
    def __init__(self, name, center, radius, height):
        super().__init__(name, [center])
        self.radius = radius
        self.height = height

    def volume(self):
        return math.pi * self.radius ** 2 * self.height / 3


Cone("traffic cone", Point(0, 0, 0), 0.2, 0.7)


TypeError: Can't instantiate abstract class Cone without an implementation for abstract method 'surface_area'

The message names the missing calculation. In the dictionary version, a cone with a missing branch
would have returned `None` from `surface_area` for as long as nobody looked. The abstract base class
turns the gap into an error the moment the first cone is made.

### TypeError: a shape built from too few points

A `Triangle` takes three points by name, so a missing one is reported when the triangle is built.


In [23]:
Triangle("roof gable", Point(0, 0), Point(6, 0))


TypeError: Triangle.__init__() missing 1 required positional argument: 'c'

`missing 1 required positional argument: 'c'` points at the call. The dictionary version accepts a
triangle with two points without a word, and fails only later, when `area` tries to unpack three of
them.

### FrozenInstanceError: moving a point in place

A `Point` is frozen, so a shape can hold one without worrying that something else will change it.


In [24]:
corner = Point(0, 0)
corner.x = 5


FrozenInstanceError: cannot assign to field 'x'

To move a point, make a new one. `replace` copies a frozen dataclass with the fields you name changed,
and leaves the original as it was.


In [25]:
moved = replace(corner, x=5)

print("original:", corner)
print("moved:   ", moved)


original: Point(x=0, y=0, z=0.0)
moved:    Point(x=5, y=0, z=0.0)


### The quiet one: two points that should be equal, and are not

A dataclass compares its fields with `==`, and the **Numbers** notebook showed that `==` on floats is
exact.


In [26]:
computed = Point(0.1 + 0.2, 0)
expected = Point(0.3, 0)

print("computed:", computed)
print("equal:   ", computed == expected)


computed: Point(x=0.30000000000000004, y=0, z=0.0)
equal:    False


The printout shows why: the computed point's `x` is `0.30000000000000004`, not `0.3`. Nothing raised,
and any code that looks for a matching point, or removes duplicate shapes, will quietly miss it.

When points or shapes are computed, compare them with a tolerance, as the **Numbers** notebook did for
single floats.


In [27]:
def same_place(a, b):
    return all(math.isclose(p, q) for p, q in [(a.x, b.x), (a.y, b.y), (a.z, b.z)])


print("same place:", same_place(computed, expected))


same place: True


## Recap

- A design decides which things become classes, what each one holds and does, and how they relate.
- The nouns in a problem suggest classes and the verbs suggest methods.
- Branching on a `kind` string spreads each new kind across every function; a class per kind keeps it
  in one place.
- A misspelled kind in a dictionary is silent. A misspelled class name is a `NameError` on that line.
- An abstract base class declares what every kind must provide, and refuses a kind that forgets.
- Composition fits parts, as a shape has points and a playlist has tracks. Inheritance fits kinds, as a
  sphere is a solid shape.
- Dataclasses suit things that are mostly data, and frozen ones suit values such as points and tracks.
- A decorator and a context manager keep retrying and all-or-nothing sending out of the classes that
  use them.
- An exception family lets callers handle each failure precisely, with the facts attached.
- For one kind of thing and one calculation, a plain function is still the better answer.
- Compare computed floats with `math.isclose`, including inside a dataclass.


## What is next

That is the end of this guide. You can write a class and explain every line of it, give it methods,
properties and the dunder methods Python calls into, build classes out of one another or on top of one
another, write down what a family of classes must provide, define the errors it can raise, and decide,
for a real problem, whether classes are worth writing at all.

The **APIs and JSON** guide comes next. It is about talking to web services: sending requests, reading
the JSON that comes back, and coping with the ways a remote service fails. It ends by building a client
you would reuse, and designing that client is the kind of decision this notebook practiced.

The guides after it move on to the libraries, and every one of them is built from classes like the ones
in this guide. When a pandas `DataFrame` gives you its `.shape` with no parentheses, you now know you are
looking at a property.


---

&#8592; **Previous:** [Exceptions as Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/13-exceptions-as-classes.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
